# Potential power creep

Reproduce original generation analysis, then challenge interpretation. This snapshot contains current stats and current chains. It cannot reconstruct historical stat buffs or prove causal power creep.

In [1]:
# Resolve project root before package imports. Works from root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "pokeml").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook inside the ml_pokemon project")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select project Python 3.14.8 kernel (.pokemon)")
print(f"Project: {PROJECT_ROOT} | Python: {sys.version.split()[0]} | Kernel: {sys.executable}")


Project: /Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon | Python: 3.14.8 | Kernel: /Users/pano/venv/pokemon/.pokemon/bin/python


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pokeml.eda.snapshot import load_analysis_snapshot

df = load_analysis_snapshot()
OUTPUT = Path("artifacts/eda/02_power_creep")
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Validated analysis snapshot: {len(df):,} rows")

Validated analysis snapshot: 1,082 rows


## Population and overall distribution

Primary analysis uses one default Pokémon per species. Selected regional/origin forms appear as sensitivity comparison. Histograms cover every included value, with left-inclusive bins.

In [3]:
from pokeml.eda.power_creep import default_species, generation_summary, fixed_composition, polynomial_trend, drift_backtest
from pokeml.eda.general import bst_histogram, get_extremes, stats_by_stage, compute_baseline
from pokeml.visualisation.viz_eda.gen_bst_plot import get_median_bst_plot
from pokeml.visualisation.viz_eda.bst_vs_stage_plot import get_bst_vs_stage_plot
from pokeml.visualisation.viz_eda.power_creep import plot_composition, plot_polynomial_hypotheses
species = default_species(df)
comparison = generation_summary(species).merge(generation_summary(df), on="generation", suffixes=("_species", "_with_forms"))
display(comparison.round(2))
comparison.to_csv(OUTPUT / "generation_summary.csv", index=False)
histogram = bst_histogram(species)
assert histogram["count"].sum() == len(species)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(histogram.lower, histogram["count"], width=20, align="edge", edgecolor="white")
ax.set(xlabel="BST; bins [lower, upper)", ylabel="Default species count", title="Complete BST distribution")
fig.savefig(OUTPUT / "bst_histogram.png", dpi=180, bbox_inches="tight")
plt.show()

,generation,count_species,species_species,median_species,mean_species,min_species,max_species,count_with_forms,species_with_forms,median_with_forms,mean_with_forms,min_with_forms,max_with_forms
0,1,151,151,405.0,407.64,195,680,151,151,405.0,407.64,195,680
1,2,100,100,415.0,407.18,180,680,100,100,415.0,407.18,180,680
2,3,135,135,410.0,403.73,190,680,135,135,410.0,403.73,190,680
3,4,107,107,480.0,445.57,194,720,107,107,480.0,445.57,194,720
4,5,156,156,445.5,425.76,255,680,156,156,445.5,425.76,255,680
5,6,72,72,450.0,429.31,200,680,72,72,450.0,429.31,200,680
6,7,88,88,480.0,449.41,175,680,106,106,475.5,439.98,175,680
7,8,96,96,481.5,439.22,180,690,133,133,483.0,446.38,180,690
8,9,120,120,488.5,457.39,210,670,122,122,488.5,455.63,210,670


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16177/4166659927.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Original overall generation plot

Overall median can rise through changing stage/rarity composition. Minima and maxima now preserve all tied Pokémon, rather than selecting one arbitrary name.

In [4]:
fig, ax = get_median_bst_plot(species, plot_path=OUTPUT)
plt.show()
maxima = pd.concat([get_extremes(species, "total_stats", out_features=["type_1", "type_2"], rarity="regular", generation=g) for g in sorted(species.generation.unique())], ignore_index=True)
display(maxima[maxima.extreme.eq("max")])

/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16177/3733555280.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,extreme,name,total_stats,generation,type_1,type_2
2,max,dragonite,600,1,dragon,flying
4,max,tyranitar,600,2,rock,dark
6,max,slaking,670,3,normal,NaN
8,max,garchomp,600,4,dragon,ground
10,max,hydreigon,600,5,dark,dragon
12,max,goodra,600,6,dragon,NaN
14,max,kommo_o,600,7,dragon,fighting
16,max,dragapult,600,8,dragon,ghost
19,max,baxcalibur,600,9,dragon,ice
20,max,archaludon,600,9,steel,dragon


## Within-stage generation trends

Enhances original panels: exact pooled baseline, sample labels and line plots suited to zoomed axes. Compare regular stages separately from legendary population.

In [5]:
configuration = {"single": (["single"], "regular"), "s1c2": (["s1c2"], "regular"), "s2c2": (["s2c2"], "regular"), "s1c3": (["s1c3"], "regular"), "s2c3": (["s2c3"], "regular"), "s3c3": (["s3c3"], "regular"), "legends": (None, "legendary")}
fig, axes = get_bst_vs_stage_plot(stats_by_stage(species, configuration), compute_baseline(species), plot_path=OUTPUT)
plt.show()
from pokeml.features.feature_initial import apply_initial_tag, get_initial_pokes
tagged = apply_initial_tag(species, get_initial_pokes())
display(tagged[tagged.initial.eq("initial")].groupby(["generation", "stage"], observed=True).total_stats.agg(["count", "median", "mean"]).round(2))
display(species[species.rarity.eq("legendary")].groupby(["generation", "stage"]).total_stats.agg(["count", "median"]))

/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16177/3808864147.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


count  median    mean
generation stage                       
1          s1c3       3   314.0  313.67
           s2c3       3   405.0  405.00
           s3c3       3   530.0  529.67
2          s1c3       3   314.0  313.67
           s2c3       3   405.0  405.00
           s3c3       3   530.0  529.67
3          s1c3       3   310.0  310.00
           s2c3       3   405.0  405.00
           s3c3       3   530.0  531.67
4          s1c3       3   314.0  313.67
           s2c3       3   405.0  405.00
           s3c3       3   530.0  529.67
5          s1c3       3   308.0  308.00
           s2c3       3   413.0  414.67
           s3c3       3   528.0  528.00
6          s1c3       3   313.0  311.33
           s2c3       3   405.0  406.33
           s3c3       3   530.0  531.33
7          s1c3       3   320.0  320.00
           s2c3       3   420.0  420.00
           s3c3       3   530.0  530.00
8          s1c3       3   310.0  310.00
           s2c3       3   420.0  420.00
           s3c3       3   530.0  530.00
9          s1c3       3   310.0  310.00
           s2c3       3   410.0  410.33
           s3c3       3   530.0  530.00

count  median
generation stage                
1          single      5   580.0
2          single      6   590.0
3          single     10   600.0
4          single     14   600.0
5          single     13   600.0
6          single      6   600.0
7          s1c2        2   417.0
           s1c3        1   200.0
           s2c2        2   585.0
           s2c3        1   400.0
           s3c3        2   680.0
           single      8   585.0
8          s1c2        1   385.0
           s2c2        1   550.0
           single     10   580.0
9          single     18   580.0

## Hold roster composition constant

Use same pooled stage weights for every generation. Compare means with means; weighted stage medians would not equal overall median. Weights are descriptive and never used in holdout forecasts. Unsupported stages would be explicitly excluded.

In [6]:
composition, weights = fixed_composition(species)
display(weights)
display(composition.round(2))
composition.to_csv(OUTPUT / "fixed_composition.csv", index=False)
regular = species[species.rarity.eq("regular")]
proportions = pd.crosstab(regular.generation, regular.stage, normalize="index")
fig, axes = plot_composition(composition, proportions)
fig.savefig(OUTPUT / "composition.png", dpi=180, bbox_inches="tight")
plt.show()

,stage,min_generation_count,supported,weight
0,s1c2,14,True,0.235676
1,s1c3,7,True,0.123243
2,s2c2,14,True,0.264865
3,s2c3,6,True,0.123243
4,s3c3,7,True,0.128649
5,single,5,True,0.124324


,generation,actual_mean,actual_median,fixed_composition_mean
0,1,400.92,405.0,409.30
1,2,393.81,410.0,400.84
2,3,386.74,405.0,389.34
3,4,419.85,450.0,407.38
4,5,409.22,418.0,414.94
5,6,411.36,417.0,416.34
6,7,429.22,469.5,412.75
7,8,419.29,470.0,407.26
8,9,435.90,479.0,417.78


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16177/2072325873.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Challenge original polynomial hypotheses

Original PDF proposed cubic F3 and linear singles from generation 3, citing fitted R² near 0.7. Recompute mean and median fits with corrected data. High fitted R² across nine generation points does not establish reliable extrapolation.

Original hypotheses are reproduced here and explained in [EDA findings](../docs/EDA_FINDINGS.md).

In [7]:
f3_fit, f3_values = polynomial_trend(species, "s3c3", degree=3)
single_fit, single_values = polynomial_trend(species, "single", degree=1, start_generation=3)
display(pd.concat([f3_fit, single_fit], ignore_index=True).round(3))
fig, axes = plot_polynomial_hypotheses({"Final three-stage: cubic hypothesis": f3_values, "Singles Gen 3+: linear hypothesis": single_values})
fig.savefig(OUTPUT / "original_polynomial_hypotheses.png", dpi=180, bbox_inches="tight")
plt.show()

,stage,statistic,degree,generations,in_sample_r2
0,s3c3,mean,3,9,0.839
1,s3c3,median,3,9,0.619
2,single,mean,1,7,0.836
3,single,median,1,7,0.728


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16177/1468727011.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Forward generation checks

Train through generations 5–8, evaluate generations 6–9. Frozen stage median competes with recent-three-generation median, linear drift and original proposals. MAE uses individual Pokémon, not only generation median. Signed error is prediction minus actual. No clipping or holdout fitting.

In [8]:
backtest = drift_backtest(species)
display(backtest.round(2))
backtest.to_csv(OUTPUT / "drift_backtest.csv", index=False)
score = backtest.groupby(["stage", "method"])[["mae", "rmse", "signed_error"]].mean()
display(score.round(2))
print("Summary weights each evaluation generation equally; inspect individual folds above")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for ax, stage in zip(axes, ["s3c3", "single"]):
    table = backtest[backtest.stage.eq(stage)].pivot(index="validation_generation", columns="method", values="mae")
    table.plot(marker="o", ax=ax)
    ax.set(title=stage, xlabel="Held-out generation", ylabel="MAE (BST)")
    ax.legend(fontsize=7)
fig.savefig(OUTPUT / "drift_holdouts.png", dpi=180)
plt.show()

,validation_generation,stage,method,train_last_generation,train_count,validation_count,prediction,mae,rmse,signed_error
0,6,s3c3,frozen_median,5,81,8,520.00,37.00,50.67,0.50
1,6,s3c3,recent_3_median,5,81,8,525.00,35.75,50.97,5.50
2,6,s3c3,linear_drift,5,81,8,531.25,34.81,52.02,11.75
3,6,s3c3,original_polynomial,5,81,8,494.90,45.58,56.33,-24.60
4,6,single,frozen_median,5,53,5,460.00,26.20,29.27,-14.60
5,6,single,recent_3_median,5,53,5,460.00,26.20,29.27,-14.60
6,6,single,linear_drift,5,53,5,443.40,36.16,40.22,-31.20
7,6,single,original_polynomial,5,53,5,484.00,22.20,27.06,9.40
8,7,s3c3,frozen_median,6,89,7,520.00,25.00,34.69,-6.43
9,7,s3c3,recent_3_median,6,89,7,525.00,24.29,34.12,-1.43


mae   rmse  signed_error
stage  method                                         
s3c3   frozen_median        25.53  35.56         -5.24
       linear_drift         25.90  35.81          7.36
       original_polynomial  39.03  44.99         13.11
       recent_3_median      24.38  35.16         -0.36
single frozen_median        47.41  57.90        -31.52
       linear_drift         48.32  58.68        -32.00
       original_polynomial  40.59  52.93         -2.34
       recent_3_median      43.69  54.13        -21.65

Summary weights each evaluation generation equally; inspect individual folds above


/var/folders/h8/kn6p4vt50_v066v_y6f9qp9w0000gn/T/ipykernel_16177/3290686629.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

Evidence can support different drift behaviour by stage. Keep results fold-specific. Cubic F3 extrapolation must beat simple alternatives before becoming model feature. These retrospective comparisons already informed by whole-dataset exploration are not untouched final tests.

## Original author notes

Original comments preserved below. They document exploration, not validated rules. Audit sections above qualify or challenge them.

### dist_general

```python
# Let us start by identifying those pokes with the lowest and greatest values for each rarity. We define first a nice func as:
# Well, well, well, well! What do we see here! Except Gen 2 and 3, all regular pokes with max total_stats are fucking dragons! Not only that! it seems that the max_stat for regular pokes is cap at 600! While the minimal one has been changing over time, but not much... 
# Here I will draw the histogram the overall distribution of pokes based on their BST. First I design a function to write BST pokemon by intervals
# Then, plot it
# ✅ Fixed rotation
# We see that majority of pokes cluster by 270-290, which seems to be related to initial stage of evolution chains. But major cluster is by 470-490. Actually, we will see that the total median is 480 BST.
# Not bad, Let us now explore how the median of each generation has changed, together with the overall median and evolution of minima and maxima. 
# First extract min each gen for regulars, and max for legends:
# Now, lets get the median for each generation
# Then, plot it
```

### Original exploratory notes

Historical comments and docstrings; current analysis above qualifies their claims.

```text
# Let us start from the .csv with the minimal amount of data
# Here we define the inputs for the function. We want to check how each mean and median distributes across generations for each pokemon stage in an evolution chain.
# And we throw a loop to print for each stage the means, medians and everything
# ← .items() gives both key and value
# This is to find the overall median per stage and category of all pokes accross all generations.
# mask creation 
# We then plot previous superdictionary for each stage, the mean and median for each generation.
# convert to array for offset math
# Two bars per generation
# Add count labels on top of mean bars
# Save it to avoid issues.
# There seems to be trends for singles and F3. Let see what happens if we fit.
# Extract the means and medians and store means and medians in a list
# [1,2,3,...,9]
# mean 
# median 
# Fit for x, which represent the generations. Last arg is the rank of the fit. Try 4
# And generate the curve for the previous fitting. Save those curves.
# From the correlation matrix, compute the square of the cross term
# And now plot. Generate a smooth x for plotting the values and the fit.
# Add your R²
# Let us check same thing for Singles. As we see, from Gen 3 onwards, we see a realistic increment for this type of pokemon. Let me manually drop first two gens:
# OBS! mns = mean(s)
# OBS! mds = medians(s)
# This is the value of the generation from which we see the incremental tendency.
# median
# And generate the linear fit for the previous fitting. Save those curves.
# Because lists start counting on 0.
```

### Original exploratory notes

Historical comments and docstrings; current analysis above qualifies their claims.

```text
interval_bst docstring:

    Basic Func to get the overall BST in intervals
    
```